In [62]:
import pandas as pd

In [64]:
df=pd.read_csv("IMDB Dataset.csv")

In [66]:
df.shape

(50000, 2)

In [68]:
df.head

<bound method NDFrame.head of                                                   review sentiment
0      One of the other reviewers has mentioned that ...  positive
1      A wonderful little production. <br /><br />The...  positive
2      I thought this was a wonderful way to spend ti...  positive
3      Basically there's a family where a little boy ...  negative
4      Petter Mattei's "Love in the Time of Money" is...  positive
...                                                  ...       ...
49995  I thought this movie did a down right good job...  positive
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  I'm going to have to disagree with the previou...  negative
49999  No one expects the Star Trek movies to be high...  negative

[50000 rows x 2 columns]>

In [71]:
df.drop_duplicates(inplace=True)

In [73]:
df.columns

Index(['review', 'sentiment'], dtype='object')

In [75]:
##preprocessing 

#converting to lowercase
df['review']=df['review'].str.lower()

In [78]:
import re



In [80]:
def remove_urls(text):
    text = re.sub(r"http\S+" , "", text)  # (pattern, repl, string) eg - https://www.google.com
    return text
    
df["review"] = df["review"].apply(remove_urls)

In [82]:
#remove puncutation

def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "", text) # A-Z a-z 0-9 \s
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [84]:
#remove html tags

def remove_html(text):
    text = re.sub(r"<.*?>" , "", text)
    return text

df["review"] = df["review"].apply(remove_html)

In [85]:
#removing stopwords
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\radhi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\radhi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\radhi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [87]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords



In [89]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [90]:
df.head()

,review,sentiment
0,e revewers nte wtchg 1 oz epoe hooke ...,positive
1,wderful ltle prducti br br filming technique...,positive
2,hugh h werful w pen me h ummer weeken n...,positive
3,bcy fly e boy jke hk zobe cloe pn f...,negative
4,peer me love i vu unng film wch mr mei ...,positive


In [92]:
#stemming

from nltk import PorterStemmer


In [94]:
def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]

    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df['review'] = df['review'].apply(stemming)

In [97]:
#encoding

from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()

df["sentiment"]=le.fit_transform(df["sentiment"])


In [99]:
y=df["sentiment"]

In [105]:
#vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer(max_features=5000)
X=tf.fit_transform(df['review'])


In [110]:
#RNN
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
    
)

In [118]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [119]:
train_set=TensorDataset(
    torch.from_numpy(X_train).float(),
     torch.from_numpy(y_train.values).float()
)

test_set=TensorDataset(
    torch.from_numpy(X_test).float(),
     torch.from_numpy(y_test.values).float()
    
)

In [121]:
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)

In [127]:
#building RNN
import torch.nn as nn
import torch.optim as optim

In [125]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out




        

In [130]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [132]:
#train and evaluate

epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")


epoch = 1/10 and loss = 0.5000250935554504
epoch = 2/10 and loss = 0.11149807274341583
epoch = 3/10 and loss = 0.2435746043920517
epoch = 4/10 and loss = 0.1916055828332901
epoch = 5/10 and loss = 0.2230691760778427
epoch = 6/10 and loss = 0.1902426928281784
epoch = 7/10 and loss = 0.22084763646125793
epoch = 8/10 and loss = 0.19901713728904724
epoch = 9/10 and loss = 0.5142643451690674
epoch = 10/10 and loss = 0.40166738629341125


In [133]:

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 82.48462236563476
